In [1]:
!pip install -q ultralytics==8.3.100 wandb

import os, yaml, json, shutil, csv
import wandb
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO, RTDETR

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
print(f"Device      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM        : {vram:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.1/977.1 kB 38.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch     : 2.10.0+cu128
CUDA        : True
Device      : Tesla T4
VRAM        : 15.6 GB


In [2]:
# We do NOT call wandb.init() here globally.
# Each training run gets its OWN wandb run so they appear as separate
# experiments in the dashboard — not one merged run.
# We define the shared config here and pass it into each run below.

WANDB_PROJECT = "NPS-Drone-Baseline-Benchmark"
WANDB_ENTITY  = None   # set to "your-username" if needed

WANDB_BASE_CONFIG = {
    "dataset"          : "NPS-Drones",
    "benchmark_source" : "TransVisDrone arXiv:2210.08423v2",
    "framework"        : "Ultralytics 8.3.100",
    "train_split"      : "videos #01-#36, all annotated frames",
    "val_split"        : "videos #37-#40, all annotated frames",
    "test_split"       : "videos #41-#50, ALL frames",
    "test_protocol"    : "every 4th frame (matches TransVisDrone + Dogfight)",
    "pretrained_on"    : "MS-COCO",
    "nc"               : 1,
    "class_names"      : ["drone"],
}

# Authenticate
from kaggle_secrets import UserSecretsClient
try:
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key)
    print("W&B authenticated via Kaggle Secret.")
except Exception:
    wandb.login()   # will prompt for key
    print("W&B authenticated via prompt.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bscs22030 (bscs22030-information-technology-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B authenticated via Kaggle Secret.


In [3]:
import yaml, os, shutil
from pathlib import Path

P = Path("/kaggle/input/datasets/kamikazeeeeeee88")

# Source dirs — images and labels are SEPARATE
SRC = {
    "train": [
        {"img": P/"nps-drone-train-part1/images_train",
         "lbl": P/"nps-drone-train-part1/labels_train"},
        {"img": P/"nps-drone-train-part2/images_train",
         "lbl": P/"nps-drone-train-part2/labels_train"},
    ],
    "val": [
        {"img": P/"nps-drone-val/images_val",
         "lbl": P/"nps-drone-val/labels_val"},
    ],
    "test": [
        {"img": P/"nps-drone-test/images_test",
         "lbl": P/"nps-drone-test/labels_test"},   # may have fewer labels than images
    ],
}

# Build a clean unified structure ultralytics understands:
#   /kaggle/working/nps/
#       images/train/   images/val/   images/test/
#       labels/train/   labels/val/   labels/test/
#
# Ultralytics infers label path by replacing 'images' → 'labels' in the path.
# This is the ONE layout that always works reliably.

DATASET_ROOT = Path("/kaggle/working/nps")

for split, src_list in SRC.items():
    img_dst = DATASET_ROOT / "images" / split
    lbl_dst = DATASET_ROOT / "labels" / split
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    n_img, n_lbl = 0, 0
    for src in src_list:
        img_src = Path(src["img"])
        lbl_src = Path(src["lbl"])

        for img_path in sorted(img_src.iterdir()):
            if img_path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                continue
            dst_img = img_dst / img_path.name
            if not dst_img.exists():
                os.symlink(img_path.resolve(), dst_img)
            n_img += 1

            lbl_path = lbl_src / img_path.with_suffix(".txt").name
            if lbl_path.exists():
                dst_lbl = lbl_dst / lbl_path.name
                if not dst_lbl.exists():
                    os.symlink(lbl_path.resolve(), dst_lbl)
                n_lbl += 1

    print(f"{split:6s}: {n_img} images | {n_lbl} labels "
          f"| {n_img - n_lbl} unannotated frames")

# Write YAML pointing to the clean structure
YAML_PATH = "/kaggle/working/nps_drone.yaml"
data_config = {
    "path" : str(DATASET_ROOT),
    "train": "images/train",
    "val"  : "images/val",
    "test" : "images/test",
    "nc"   : 1,
    "names": {0: "drone"},
}
with open(YAML_PATH, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"\nYAML written: {YAML_PATH}")
print(open(YAML_PATH).read())

train : 32220 images | 32220 labels | 0 unannotated frames
val   : 3753 images | 3753 labels | 0 unannotated frames
test  : 12355 images | 12355 labels | 0 unannotated frames

YAML written: /kaggle/working/nps_drone.yaml
names:
  0: drone
nc: 1
path: /kaggle/working/nps
test: images/test
train: images/train
val: images/val



In [4]:
# ── Frozen training config — DO NOT change between models ─────────────────
# Every baseline uses identical settings so differences = architecture only.

CONFIG = {
    # ── Core ────────────────────────────────────────────────────────────
    "imgsz"          : 640,
    "epochs"         : 100,
    "batch"          : 32,       # auto-batch (~60% VRAM)
    "workers"        : 4,
    "seed"           : 42,
    "pretrained"     : True,     # COCO weights — matches paper
    "single_cls"     : True,     # single-class (drone)

    # ── Optimizer — matches TransVisDrone ────────────────────────────────
    # Paper: Adam, lr=3e-5, momentum=0.843
    # We use AdamW (strictly better Adam variant) at same lr/momentum
    "optimizer"      : "AdamW",
    "lr0"            : 3e-5,     # exact match to paper
    "lrf"            : 0.1,      # final_lr = lr0 * lrf = 3e-6
    "momentum"       : 0.843,    # exact match to paper
    "weight_decay"   : 0.0005,
    "warmup_epochs"  : 3,
    "warmup_momentum": 0.8,
    "cos_lr"         : True,     # cosine decay — matches paper

    # ── Augmentation (Zhu et al. TPH-YOLOv5 strengths for small objects) ─
    # Paper cites [39] Zhu et al. for augmentation strengths
    "augment"        : True,
    "mosaic"         : 1.0,
    "hsv_h"          : 0.015,
    "hsv_s"          : 0.7,
    "hsv_v"          : 0.4,
    "translate"      : 0.1,
    "scale"          : 0.5,
    "fliplr"         : 0.5,
    "flipud"         : 0.0,
    "degrees"        : 0.0,      # no rotation — bboxes are axis-aligned
    "perspective"    : 0.0001,   # slight perspective (paper uses it)
    "shear"          : 0.0,
    "mixup"          : 0.0,

    # ── Regularisation ──────────────────────────────────────────────────
    "patience"       : 30,       # early stopping

    # ── Eval thresholds — exact match to paper ───────────────────────────
    "conf"           : 0.001,    # confidence threshold
    "iou"            : 0.6,      # NMS IoU threshold

    # ── Checkpointing ───────────────────────────────────────────────────
    "save"           : True,
    "save_period"    : 10,       # save checkpoint every 10 epochs
    "val"            : True,
    "plots"          : True,
    "verbose"        : True,
}

# Metrics we will report — mirrors TransVisDrone paper Tables I & II
METRICS_TO_REPORT = [
    "AP@0.5",          # primary metric — Table I, II
    "Precision",       # at best F1 threshold
    "Recall",          # at best F1 threshold
    "FPS",             # throughput (frames/sec)
    "FPPI",            # False Positives Per Image — Section V-B
    "AP@0.5:0.95",     # COCO standard (not in paper, useful for context)
    "F1",              # derived from P and R
]

# Reference numbers from TransVisDrone Table I (for comparison)
TRANSVISDRONE_NPS = {
    "model"      : "TransVisDrone (τ=5, 1280px)",
    "AP@0.5"     : 0.95,
    "Precision"  : 0.92,
    "Recall"     : 0.91,
    "FPS"        : 24.6,
    "FPPI"       : 0.000437,
    "AP@0.5:0.95": "N/R",    # not reported
    "F1"         : "N/R",
}

print("CONFIG locked. Key parameters vs TransVisDrone paper:")
print(f"  lr0       : {CONFIG['lr0']}  (paper: 3e-5  ✅)")
print(f"  momentum  : {CONFIG['momentum']}  (paper: 0.843 ✅)")
print(f"  cos_lr    : {CONFIG['cos_lr']}   (paper: True  ✅)")
print(f"  conf      : {CONFIG['conf']}  (paper: 0.001 ✅)")
print(f"  iou (NMS) : {CONFIG['iou']}    (paper: 0.6   ✅)")
print(f"  pretrained: {CONFIG['pretrained']}  (paper: COCO  ✅)")

CONFIG locked. Key parameters vs TransVisDrone paper:
  lr0       : 3e-05  (paper: 3e-5  ✅)
  momentum  : 0.843  (paper: 0.843 ✅)
  cos_lr    : True   (paper: True  ✅)
  conf      : 0.001  (paper: 0.001 ✅)
  iou (NMS) : 0.6    (paper: 0.6   ✅)
  pretrained: True  (paper: COCO  ✅)


In [5]:
# ── Evaluation helpers ─────────────────────────────────────────────────────

def get_every_4th_frame_paths(test_img_dir: str) -> list:
    """
    Return every 4th frame from the test directory, sorted by filename.
    Matches TransVisDrone evaluation protocol (skip rate = 4).
    Frames are sorted alphanumerically — assumes filenames encode order
    (e.g. frame_000001.png, frame_000002.png ...).
    """
    p = Path(test_img_dir)
    all_frames = sorted(
        list(p.glob("*.png")) + list(p.glob("*.jpg"))
    )
    # every 4th: index 0, 4, 8, 12 ...
    subset = all_frames[::4]
    print(f"Test set  : {len(all_frames)} total frames")
    print(f"Evaluating: {len(subset)} frames (every 4th, skip=4)")
    return subset


def compute_fppi(pred_label_dir: Path, test_img_dir: str,
                 evaluated_stems: set) -> float:
    """
    FPPI = total false positives / total evaluated frames.
    A prediction on a frame with no ground-truth label = false positive.
    Ultralytics saves per-image prediction .txt files when save_txt=True.

    Args:
        pred_label_dir  : path to ultralytics labels output dir
        test_img_dir    : test image directory (to find all stems)
        evaluated_stems : set of frame stems that were evaluated (every-4th)
    """
    test_path  = Path(test_img_dir)
    all_imgs   = sorted(list(test_path.glob("*.png")) + list(test_path.glob("*.jpg")))

    # Frames that have a ground-truth label (non-empty .txt alongside image)
    frames_with_gt = set()
    for img in all_imgs:
        lbl = img.with_suffix(".txt")
        if lbl.exists() and lbl.stat().st_size > 0:
            frames_with_gt.add(img.stem)

    empty_stems = evaluated_stems - frames_with_gt
    total_evaluated = len(evaluated_stems)
    total_fp = 0

    if not pred_label_dir.exists():
        print("  ⚠️  No prediction label directory found. "
              "Re-run val() with save_txt=True.")
        return None

    for stem in empty_stems:
        pred_file = pred_label_dir / f"{stem}.txt"
        if pred_file.exists():
            with open(pred_file) as f:
                n_preds = sum(1 for line in f if line.strip())
            total_fp += n_preds

    fppi = total_fp / total_evaluated if total_evaluated > 0 else 0.0
    print(f"  FPPI = {fppi:.6f}  "
          f"({total_fp} FPs on {len(empty_stems)} empty frames "
          f"/ {total_evaluated} evaluated frames)")
    return fppi


def extract_metrics(val_results, model_name: str) -> dict:
    """Extract all metrics we need to report, matching TransVisDrone paper."""
    box = val_results.box
    ap50     = float(box.ap50.mean()) if hasattr(box.ap50, "mean") else float(box.ap50)
    ap50_95  = float(box.map)
    prec     = float(box.mp)
    rec      = float(box.mr)
    f1       = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    speed    = val_results.speed
    total_ms = sum(speed.values())
    fps      = round(1000.0 / total_ms, 2) if total_ms > 0 else 0.0

    m = {
        "model"       : model_name,
        "AP@0.5"      : round(ap50, 4),
        "AP@0.5:0.95" : round(ap50_95, 4),
        "Precision"   : round(prec, 4),
        "Recall"      : round(rec, 4),
        "F1"          : round(f1, 4),
        "FPS"         : fps,
        "inf_ms"      : round(speed.get("inference", 0), 2),
        "FPPI"        : None,   # filled in after compute_fppi()
    }

    print(f"\n  ┌── {model_name} ─────────────────────────")
    for k, v in m.items():
        print(f"  │  {k:<14s}: {v}")
    print(f"  └────────────────────────────────────────")
    return m


# Running results log — persists across all model cells
ALL_RESULTS  = [TRANSVISDRONE_NPS]   # seed with paper reference
RESULTS_PATH = Path("/kaggle/working/baseline_results.csv")

def save_results_csv():
    cols = ["model","AP@0.5","AP@0.5:0.95","Precision","Recall","F1","FPS","FPPI","inf_ms"]
    with open(RESULTS_PATH, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader(); w.writerows(ALL_RESULTS)
    print(f"Results saved → {RESULTS_PATH}")

def print_results_table():
    import pandas as pd
    df = pd.DataFrame(ALL_RESULTS)
    cols = ["model","AP@0.5","Precision","Recall","F1","FPS","FPPI"]
    cols = [c for c in cols if c in df.columns]
    print("\n" + "="*75)
    print("RESULTS vs TransVisDrone — NPS Dataset")
    print("="*75)
    print(df[cols].to_string(index=False))
    print("="*75)

print("Helpers defined: get_every_4th_frame_paths, compute_fppi, "
      "extract_metrics, save_results_csv, print_results_table")

Helpers defined: get_every_4th_frame_paths, compute_fppi, extract_metrics, save_results_csv, print_results_table


In [6]:
# ── Custom W&B Callbacks — real-time logging + periodic weight uploads ─────
from ultralytics.utils import callbacks as ul_callbacks

WEIGHT_UPLOAD_EVERY_N_EPOCHS = 3

def on_train_epoch_end(trainer):
    """Log train losses + LR in real time after every epoch."""
    try:
        loss_items = trainer.loss_items
        metrics = {
            "train/box_loss": float(loss_items[0]),
            "train/cls_loss": float(loss_items[1]),
            "train/dfl_loss": float(loss_items[2]),
            "train/lr"      : trainer.optimizer.param_groups[0]["lr"],
            "epoch"         : trainer.epoch + 1,
        }
        if wandb.run is not None:
            wandb.log(metrics, step=trainer.epoch + 1)
    except Exception as e:
        print(f"  ⚠️  on_train_epoch_end: {e}")


def on_fit_epoch_end(trainer):
    """Log val metrics + upload weights every N epochs."""
    epoch = trainer.epoch + 1
    try:
        if hasattr(trainer, "metrics") and trainer.metrics:
            val_metrics = {
                f"val/{k}": float(v)
                for k, v in trainer.metrics.items()
                if isinstance(v, (int, float))
            }
            val_metrics["epoch"] = epoch
            if wandb.run is not None:
                wandb.log(val_metrics, step=epoch)
    except Exception as e:
        print(f"  ⚠️  on_fit_epoch_end (val log): {e}")

    # Periodic weight upload
    if epoch % WEIGHT_UPLOAD_EVERY_N_EPOCHS == 0:
        try:
            last_pt = Path(trainer.save_dir) / "weights" / "last.pt"
            if last_pt.exists() and wandb.run is not None:
                meta = {"epoch": epoch}
                if hasattr(trainer, "metrics"):
                    meta.update({k: float(v) for k, v in trainer.metrics.items()
                                 if isinstance(v, (int, float))})
                artifact = wandb.Artifact(
                    name     = f"{trainer.args.name}-epoch-{epoch:03d}",
                    type     = "model",
                    metadata = meta,
                )
                artifact.add_file(str(last_pt), name="last.pt")
                wandb.run.log_artifact(artifact)
                print(f"  📦 Weights uploaded → W&B artifact: epoch {epoch}")
        except Exception as e:
            print(f"  ⚠️  on_fit_epoch_end (upload): {e}")


def on_train_end(trainer):
    """Upload final best.pt."""
    try:
        best_pt = Path(trainer.save_dir) / "weights" / "best.pt"
        if best_pt.exists() and wandb.run is not None:
            artifact = wandb.Artifact(
                name     = f"{trainer.args.name}-best-final",
                type     = "model",
                metadata = {"source": "best.pt"},
            )
            artifact.add_file(str(best_pt), name="best.pt")
            wandb.run.log_artifact(artifact)
            print("  📦 Final best.pt uploaded → W&B artifact.")
    except Exception as e:
        print(f"  ⚠️  on_train_end: {e}")


# Clear any stale registrations if re-running this cell
for event, fn in [
    ("on_train_epoch_end", on_train_epoch_end),
    ("on_fit_epoch_end",   on_fit_epoch_end),
    ("on_train_end",       on_train_end),
]:
    lst = ul_callbacks.default_callbacks[event]
    if fn not in lst:
        lst.append(fn)

print(f"✅  Callbacks registered.")
print(f"    → Train loss + LR : every epoch (real-time)")
print(f"    → Val metrics     : every epoch")
print(f"    → Weight upload   : every {WEIGHT_UPLOAD_EVERY_N_EPOCHS} epochs")
print(f"    → Final best.pt   : on train end")

✅  Callbacks registered.
    → Train loss + LR : every epoch (real-time)
    → Val metrics     : every epoch
    → Weight upload   : every 3 epochs
    → Final best.pt   : on train end


In [7]:
# ── Disable ultralytics' built-in W&B callback (do this BEFORE YOLO()) ────
from ultralytics.utils import callbacks as ul_callbacks

# Remove the default wandb callback if present
for event in ul_callbacks.default_callbacks:
    ul_callbacks.default_callbacks[event] = [
        fn for fn in ul_callbacks.default_callbacks[event]
        if "wandb" not in getattr(fn, "__module__", "").lower()
    ]

print("✅ Built-in W&B callback stripped. Our custom callbacks remain.")

✅ Built-in W&B callback stripped. Our custom callbacks remain.


In [ ]:
# ── YOLO11s @ 640 — Baseline 2 (resume-aware) ─────────────────────────────
MODEL_ID   = "yolo11s_640"
WEIGHTS    = "yolo11s.pt"
RUN_NAME   = f"{MODEL_ID}_baseline"

# ── Resume logic ───────────────────────────────────────────────────────────
_run_dir      = Path("NPS-Drone-Baseline-Benchmark") / RUN_NAME
_last_pt      = _run_dir / "weights" / "last.pt"
RESUME        = _last_pt.exists()
TRAIN_WEIGHTS = str(_last_pt) if RESUME else WEIGHTS
print(f"{'⚡  Resuming from' if RESUME else '🚀  Fresh run from'} {TRAIN_WEIGHTS}")

# ── 1. Init W&B run ────────────────────────────────────────────────────────
run = wandb.init(
    project  = WANDB_PROJECT,
    entity   = WANDB_ENTITY,
    name     = RUN_NAME,
    group    = "spatial-only-640",
    job_type = "train",
    tags     = ["nps", "baseline", "640", "spatial-only", MODEL_ID],
    config   = {**WANDB_BASE_CONFIG, **CONFIG,
                "model_id": MODEL_ID, "weights": WEIGHTS,
                "resumed": RESUME},
    resume   = "allow",
    reinit   = True,
)# ── YOLO11s @ 640 — Baseline 2 (resume-aware) ─────────────────────────────
MODEL_ID   = "yolo11s_640"
WEIGHTS    = "yolo11s.pt"
RUN_NAME   = f"{MODEL_ID}_baseline"

# ── Resume logic ───────────────────────────────────────────────────────────
_run_dir      = Path("NPS-Drone-Baseline-Benchmark") / RUN_NAME
_last_pt      = _run_dir / "weights" / "last.pt"
RESUME        = _last_pt.exists()
TRAIN_WEIGHTS = str(_last_pt) if RESUME else WEIGHTS
print(f"{'⚡  Resuming from' if RESUME else '🚀  Fresh run from'} {TRAIN_WEIGHTS}")

# ── 1. Init W&B run ────────────────────────────────────────────────────────
run = wandb.init(
    project  = WANDB_PROJECT,
    entity   = WANDB_ENTITY,
    name     = RUN_NAME,
    group    = "spatial-only-640",
    job_type = "train",
    tags     = ["nps", "baseline", "640", "spatial-only", MODEL_ID],
    config   = {**WANDB_BASE_CONFIG, **CONFIG,
                "model_id": MODEL_ID, "weights": WEIGHTS,
                "resumed": RESUME},
    resume   = "allow",
    reinit   = True,
)

# ── 2. Define callbacks ────────────────────────────────────────────────────
WEIGHT_UPLOAD_EVERY_N_EPOCHS = 3

def on_train_epoch_end(trainer):
    try:
        metrics = {
            "train/box_loss": float(trainer.loss_items[0]),
            "train/cls_loss": float(trainer.loss_items[1]),
            "train/dfl_loss": float(trainer.loss_items[2]),
            "train/lr"      : trainer.optimizer.param_groups[0]["lr"],
            "epoch"         : trainer.epoch + 1,
        }
        if wandb.run is not None:
            wandb.log(metrics, step=trainer.epoch + 1)
            print(f"  📊 W&B logged train metrics epoch {trainer.epoch + 1}")
    except Exception as e:
        print(f"  ⚠️  on_train_epoch_end: {e}")

def on_fit_epoch_end(trainer):
    epoch = trainer.epoch + 1
    try:
        if hasattr(trainer, "metrics") and trainer.metrics:
            val_metrics = {
                f"val/{k}": float(v)
                for k, v in trainer.metrics.items()
                if isinstance(v, (int, float))
            }
            val_metrics["epoch"] = epoch
            if wandb.run is not None:
                wandb.log(val_metrics, step=epoch)
                print(f"  📊 W&B logged val metrics epoch {epoch}: {val_metrics}")
    except Exception as e:
        print(f"  ⚠️  on_fit_epoch_end (val): {e}")

    if epoch % WEIGHT_UPLOAD_EVERY_N_EPOCHS == 0:
        try:
            last_pt = Path(trainer.save_dir) / "weights" / "last.pt"
            if last_pt.exists() and wandb.run is not None:
                meta = {"epoch": epoch}
                if hasattr(trainer, "metrics"):
                    meta.update({k: float(v) for k, v in trainer.metrics.items()
                                 if isinstance(v, (int, float))})
                art = wandb.Artifact(
                    name=f"{trainer.args.name}-epoch-{epoch:03d}",
                    type="model", metadata=meta,
                )
                art.add_file(str(last_pt), name="last.pt")
                wandb.run.log_artifact(art)
                print(f"  📦 Weights uploaded → epoch {epoch}")
        except Exception as e:
            print(f"  ⚠️  on_fit_epoch_end (upload): {e}")

def on_train_end(trainer):
    try:
        best_pt = Path(trainer.save_dir) / "weights" / "best.pt"
        if best_pt.exists() and wandb.run is not None:
            art = wandb.Artifact(
                name=f"{trainer.args.name}-best-final",
                type="model", metadata={"source": "best.pt"},
            )
            art.add_file(str(best_pt), name="best.pt")
            wandb.run.log_artifact(art)
            print("  📦 Final best.pt uploaded.")
    except Exception as e:
        print(f"  ⚠️  on_train_end: {e}")

# ── 3. Init model, strip built-in W&B callback, add ours ──────────────────
model = YOLO(TRAIN_WEIGHTS)

# Strip only the built-in wandb callbacks from THIS model instance
for event in list(model.callbacks.keys()):
    model.callbacks[event] = [
        fn for fn in model.callbacks[event]
        if "wandb" not in getattr(fn, "__module__", "").lower()
    ]

# Add our callbacks to THIS model instance
model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_fit_epoch_end",   on_fit_epoch_end)
model.add_callback("on_train_end",       on_train_end)

print("✅ Callbacks registered on model instance:")
print(f"   on_train_epoch_end: {[f.__name__ for f in model.callbacks['on_train_epoch_end']]}")
print(f"   on_fit_epoch_end  : {[f.__name__ for f in model.callbacks['on_fit_epoch_end']]}")
print(f"   on_train_end      : {[f.__name__ for f in model.callbacks['on_train_end']]}")

# ── 4. Train ───────────────────────────────────────────────────────────────
train_results = model.train(
    data      = YAML_PATH,

    imgsz     = CONFIG["imgsz"],
    epochs    = CONFIG["epochs"],
    batch     = CONFIG["batch"],
    workers   = CONFIG["workers"],
    seed      = CONFIG["seed"],
    pretrained= CONFIG["pretrained"],
    single_cls= CONFIG["single_cls"],

    optimizer      = CONFIG["optimizer"],
    lr0            = CONFIG["lr0"],
    lrf            = CONFIG["lrf"],
    momentum       = CONFIG["momentum"],
    weight_decay   = CONFIG["weight_decay"],
    warmup_epochs  = CONFIG["warmup_epochs"],
    warmup_momentum= CONFIG["warmup_momentum"],
    cos_lr         = CONFIG["cos_lr"],

    augment    = CONFIG["augment"],
    mosaic     = CONFIG["mosaic"],
    hsv_h      = CONFIG["hsv_h"],
    hsv_s      = CONFIG["hsv_s"],
    hsv_v      = CONFIG["hsv_v"],
    translate  = CONFIG["translate"],
    scale      = CONFIG["scale"],
    fliplr     = CONFIG["fliplr"],
    flipud     = CONFIG["flipud"],
    degrees    = CONFIG["degrees"],
    perspective= CONFIG["perspective"],
    shear      = CONFIG["shear"],
    mixup      = CONFIG["mixup"],

    patience   = CONFIG["patience"],

    project    = "NPS-Drone-Baseline-Benchmark",
    name       = RUN_NAME,
    save       = CONFIG["save"],
    save_period= CONFIG["save_period"],
    val        = CONFIG["val"],
    plots      = CONFIG["plots"],
    verbose    = CONFIG["verbose"],
    exist_ok   = True,
    resume     = RESUME,
)

# ── 5. Evaluate on every-4th-frame test subset ────────────────────────────
every4th        = get_every_4th_frame_paths(TEST_DIR)
evaluated_stems = {p.stem for p in every4th}

tmp_test     = Path("/kaggle/working/tmp_test_every4th/images")
tmp_test_lbl = Path("/kaggle/working/tmp_test_every4th/labels")
tmp_test.mkdir(parents=True, exist_ok=True)
tmp_test_lbl.mkdir(parents=True, exist_ok=True)

for src in every4th:
    dst = tmp_test / src.name
    if not dst.exists():
        os.symlink(src.resolve(), dst)
    lbl_src = src.with_suffix(".txt")
    lbl_dst = tmp_test_lbl / src.with_suffix(".txt").name
    if lbl_src.exists() and not lbl_dst.exists():
        os.symlink(lbl_src.resolve(), lbl_dst)

subset_yaml = {
    "path" : "/kaggle/working/tmp_test_every4th",
    "train": "images",
    "val"  : "images",
    "test" : "images",
    "nc": 1, "names": {0: "drone"},
}
SUBSET_YAML = "/kaggle/working/subset_test.yaml"
with open(SUBSET_YAML, "w") as f:
    yaml.dump(subset_yaml, f)

best_weights = _run_dir / "weights" / "best.pt"
eval_model   = YOLO(str(best_weights))

print(f"\n--- Research Evaluation: {MODEL_ID} on every-4th-frame test set ---")
test_metrics_raw = eval_model.val(
    data     = SUBSET_YAML,
    split    = "test",
    imgsz    = CONFIG["imgsz"],
    conf     = CONFIG["conf"],
    iou      = CONFIG["iou"],
    save_json= True,
    save_txt = True,
    plots    = True,
    verbose  = True,
)

# ── 6. Extract + compute FPPI ──────────────────────────────────────────────
metrics = extract_metrics(test_metrics_raw, MODEL_ID)
pred_lbl_dir    = _run_dir / "labels"
metrics["FPPI"] = compute_fppi(pred_lbl_dir, TEST_DIR, evaluated_stems)

# ── 7. Log test metrics + plots to W&B ────────────────────────────────────
if wandb.run is None:
    run = wandb.init(project=WANDB_PROJECT, name=RUN_NAME, resume="allow", reinit=True)

wandb.log({
    "test/AP50"      : metrics["AP@0.5"],
    "test/AP50_95"   : metrics["AP@0.5:0.95"],
    "test/Precision" : metrics["Precision"],
    "test/Recall"    : metrics["Recall"],
    "test/F1"        : metrics["F1"],
    "test/FPS"       : metrics["FPS"],
    "test/FPPI"      : metrics["FPPI"] or 0,
    "test/inf_ms"    : metrics["inf_ms"],
})

for plot in ["PR_curve.png", "results.png", "confusion_matrix.png", "F1_curve.png"]:
    p = _run_dir / plot
    if p.exists():
        wandb.log({plot.replace(".png", ""): wandb.Image(str(p))})

artifact = wandb.Artifact(
    name     = f"{MODEL_ID}-weights",
    type     = "model",
    metadata = metrics,
)
for wname in ["best.pt", "last.pt"]:
    wp = _run_dir / "weights" / wname
    if wp.exists():
        artifact.add_file(str(wp), name=wname)
run.log_artifact(artifact, aliases=["best", MODEL_ID])

# ── 8. Local results table ─────────────────────────────────────────────────
ALL_RESULTS.append(metrics)
save_results_csv()
print_results_table()

wandb.finish()
print(f"\n✅  {MODEL_ID} complete.")

shutil.rmtree("/kaggle/working/tmp_test_every4th", ignore_errors=True)

# ── 2. Define callbacks ────────────────────────────────────────────────────
WEIGHT_UPLOAD_EVERY_N_EPOCHS = 3

def on_train_epoch_end(trainer):
    try:
        metrics = {
            "train/box_loss": float(trainer.loss_items[0]),
            "train/cls_loss": float(trainer.loss_items[1]),
            "train/dfl_loss": float(trainer.loss_items[2]),
            "train/lr"      : trainer.optimizer.param_groups[0]["lr"],
            "epoch"         : trainer.epoch + 1,
        }
        if wandb.run is not None:
            wandb.log(metrics, step=trainer.epoch + 1)
            print(f"  📊 W&B logged train metrics epoch {trainer.epoch + 1}")
    except Exception as e:
        print(f"  ⚠️  on_train_epoch_end: {e}")

def on_fit_epoch_end(trainer):
    epoch = trainer.epoch + 1
    try:
        if hasattr(trainer, "metrics") and trainer.metrics:
            val_metrics = {
                f"val/{k}": float(v)
                for k, v in trainer.metrics.items()
                if isinstance(v, (int, float))
            }
            val_metrics["epoch"] = epoch
            if wandb.run is not None:
                wandb.log(val_metrics, step=epoch)
                print(f"  📊 W&B logged val metrics epoch {epoch}: {val_metrics}")
    except Exception as e:
        print(f"  ⚠️  on_fit_epoch_end (val): {e}")

    if epoch % WEIGHT_UPLOAD_EVERY_N_EPOCHS == 0:
        try:
            last_pt = Path(trainer.save_dir) / "weights" / "last.pt"
            if last_pt.exists() and wandb.run is not None:
                meta = {"epoch": epoch}
                if hasattr(trainer, "metrics"):
                    meta.update({k: float(v) for k, v in trainer.metrics.items()
                                 if isinstance(v, (int, float))})
                art = wandb.Artifact(
                    name=f"{trainer.args.name}-epoch-{epoch:03d}",
                    type="model", metadata=meta,
                )
                art.add_file(str(last_pt), name="last.pt")
                wandb.run.log_artifact(art)
                print(f"  📦 Weights uploaded → epoch {epoch}")
        except Exception as e:
            print(f"  ⚠️  on_fit_epoch_end (upload): {e}")

def on_train_end(trainer):
    try:
        best_pt = Path(trainer.save_dir) / "weights" / "best.pt"
        if best_pt.exists() and wandb.run is not None:
            art = wandb.Artifact(
                name=f"{trainer.args.name}-best-final",
                type="model", metadata={"source": "best.pt"},
            )
            art.add_file(str(best_pt), name="best.pt")
            wandb.run.log_artifact(art)
            print("  📦 Final best.pt uploaded.")
    except Exception as e:
        print(f"  ⚠️  on_train_end: {e}")

# ── 3. Init model, strip built-in W&B callback, add ours ──────────────────
model = YOLO(TRAIN_WEIGHTS)

# Strip only the built-in wandb callbacks from THIS model instance
for event in list(model.callbacks.keys()):
    model.callbacks[event] = [
        fn for fn in model.callbacks[event]
        if "wandb" not in getattr(fn, "__module__", "").lower()
    ]

# Add our callbacks to THIS model instance
model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_fit_epoch_end",   on_fit_epoch_end)
model.add_callback("on_train_end",       on_train_end)

print("✅ Callbacks registered on model instance:")
print(f"   on_train_epoch_end: {[f.__name__ for f in model.callbacks['on_train_epoch_end']]}")
print(f"   on_fit_epoch_end  : {[f.__name__ for f in model.callbacks['on_fit_epoch_end']]}")
print(f"   on_train_end      : {[f.__name__ for f in model.callbacks['on_train_end']]}")

# ── 4. Train ───────────────────────────────────────────────────────────────
train_results = model.train(
    data      = YAML_PATH,

    imgsz     = CONFIG["imgsz"],
    epochs    = CONFIG["epochs"],
    batch     = CONFIG["batch"],
    workers   = CONFIG["workers"],
    seed      = CONFIG["seed"],
    pretrained= CONFIG["pretrained"],
    single_cls= CONFIG["single_cls"],

    optimizer      = CONFIG["optimizer"],
    lr0            = CONFIG["lr0"],
    lrf            = CONFIG["lrf"],
    momentum       = CONFIG["momentum"],
    weight_decay   = CONFIG["weight_decay"],
    warmup_epochs  = CONFIG["warmup_epochs"],
    warmup_momentum= CONFIG["warmup_momentum"],
    cos_lr         = CONFIG["cos_lr"],

    augment    = CONFIG["augment"],
    mosaic     = CONFIG["mosaic"],
    hsv_h      = CONFIG["hsv_h"],
    hsv_s      = CONFIG["hsv_s"],
    hsv_v      = CONFIG["hsv_v"],
    translate  = CONFIG["translate"],
    scale      = CONFIG["scale"],
    fliplr     = CONFIG["fliplr"],
    flipud     = CONFIG["flipud"],
    degrees    = CONFIG["degrees"],
    perspective= CONFIG["perspective"],
    shear      = CONFIG["shear"],
    mixup      = CONFIG["mixup"],

    patience   = CONFIG["patience"],

    project    = "NPS-Drone-Baseline-Benchmark",
    name       = RUN_NAME,
    save       = CONFIG["save"],
    save_period= CONFIG["save_period"],
    val        = CONFIG["val"],
    plots      = CONFIG["plots"],
    verbose    = CONFIG["verbose"],
    exist_ok   = True,
    resume     = RESUME,
)

# ── 5. Evaluate on every-4th-frame test subset ────────────────────────────
every4th        = get_every_4th_frame_paths(TEST_DIR)
evaluated_stems = {p.stem for p in every4th}

tmp_test     = Path("/kaggle/working/tmp_test_every4th/images")
tmp_test_lbl = Path("/kaggle/working/tmp_test_every4th/labels")
tmp_test.mkdir(parents=True, exist_ok=True)
tmp_test_lbl.mkdir(parents=True, exist_ok=True)

for src in every4th:
    dst = tmp_test / src.name
    if not dst.exists():
        os.symlink(src.resolve(), dst)
    lbl_src = src.with_suffix(".txt")
    lbl_dst = tmp_test_lbl / src.with_suffix(".txt").name
    if lbl_src.exists() and not lbl_dst.exists():
        os.symlink(lbl_src.resolve(), lbl_dst)

subset_yaml = {
    "path" : "/kaggle/working/tmp_test_every4th",
    "train": "images",
    "val"  : "images",
    "test" : "images",
    "nc": 1, "names": {0: "drone"},
}
SUBSET_YAML = "/kaggle/working/subset_test.yaml"
with open(SUBSET_YAML, "w") as f:
    yaml.dump(subset_yaml, f)

best_weights = _run_dir / "weights" / "best.pt"
eval_model   = YOLO(str(best_weights))

print(f"\n--- Research Evaluation: {MODEL_ID} on every-4th-frame test set ---")
test_metrics_raw = eval_model.val(
    data     = SUBSET_YAML,
    split    = "test",
    imgsz    = CONFIG["imgsz"],
    conf     = CONFIG["conf"],
    iou      = CONFIG["iou"],
    save_json= True,
    save_txt = True,
    plots    = True,
    verbose  = True,
)

# ── 6. Extract + compute FPPI ──────────────────────────────────────────────
metrics = extract_metrics(test_metrics_raw, MODEL_ID)
pred_lbl_dir    = _run_dir / "labels"
metrics["FPPI"] = compute_fppi(pred_lbl_dir, TEST_DIR, evaluated_stems)

# ── 7. Log test metrics + plots to W&B ────────────────────────────────────
if wandb.run is None:
    run = wandb.init(project=WANDB_PROJECT, name=RUN_NAME, resume="allow", reinit=True)

wandb.log({
    "test/AP50"      : metrics["AP@0.5"],
    "test/AP50_95"   : metrics["AP@0.5:0.95"],
    "test/Precision" : metrics["Precision"],
    "test/Recall"    : metrics["Recall"],
    "test/F1"        : metrics["F1"],
    "test/FPS"       : metrics["FPS"],
    "test/FPPI"      : metrics["FPPI"] or 0,
    "test/inf_ms"    : metrics["inf_ms"],
})

for plot in ["PR_curve.png", "results.png", "confusion_matrix.png", "F1_curve.png"]:
    p = _run_dir / plot
    if p.exists():
        wandb.log({plot.replace(".png", ""): wandb.Image(str(p))})

artifact = wandb.Artifact(
    name     = f"{MODEL_ID}-weights",
    type     = "model",
    metadata = metrics,
)
for wname in ["best.pt", "last.pt"]:
    wp = _run_dir / "weights" / wname
    if wp.exists():
        artifact.add_file(str(wp), name=wname)
run.log_artifact(artifact, aliases=["best", MODEL_ID])

# ── 8. Local results table ─────────────────────────────────────────────────
ALL_RESULTS.append(metrics)
save_results_csv()
print_results_table()

wandb.finish()
print(f"\n✅  {MODEL_ID} complete.")

shutil.rmtree("/kaggle/working/tmp_test_every4th", ignore_errors=True)

⚡  Resuming from NPS-Drone-Baseline-Benchmark/yolo11s_640_baseline/weights/last.pt


epoch,▁
train/box_loss,▁
train/cls_loss,▁
train/dfl_loss,▁
train/lr,▁
val/metrics/mAP50(B),▁
val/metrics/mAP50-95(B),▁
val/metrics/precision(B),▁
val/metrics/recall(B),▁
val/val/box_loss,▁
+2,...


✅ Callbacks registered on model instance:
   on_train_epoch_end: ['on_train_epoch_end', 'on_train_epoch_end', 'on_train_epoch_end']
   on_fit_epoch_end  : ['on_fit_epoch_end', 'on_fit_epoch_end', 'on_fit_epoch_end']
   on_train_end      : ['on_train_end', 'on_train_end', 'on_train_end']
New https://pypi.org/project/ultralytics/8.4.50 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.100 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=NPS-Drone-Baseline-Benchmark/yolo11s_640_baseline/weights/last.pt, data=/kaggle/working/nps_drone.yaml, epochs=100, time=None, patience=30, batch=32, imgsz=640, save=True, save_period=10, cache=False, device=None, workers=4, project=NPS-Drone-Baseline-Benchmark, name=yolo11s_640_baseline, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=True, rect=False, cos_lr=True, close_mosaic=10, resume=NPS-Drone-Baseline-Benchmark/yol

train: Scanning /kaggle/working/nps/labels/train.cache... 32220 images, 0 backgrounds, 0 corrupt: 100%|██████████| 32220/32220 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

  📊 W&B logged train metrics epoch 38


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 59/59 [01:06<00:00,  1.12s/it]

                   all       3753       4657      0.843      0.763      0.869      0.427


  📊 W&B logged val metrics epoch 38: {'val/metrics/precision(B)': 0.84254, 'val/metrics/recall(B)': 0.76294, 'val/metrics/mAP50(B)': 0.86922, 'val/metrics/mAP50-95(B)': 0.42667, 'val/val/box_loss': 1.81246, 'val/val/cls_loss': 0.92952, 'val/val/dfl_loss': 0.80792, 'epoch': 38}

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     39/100      8.48G      1.503     0.6429     0.7589         57        640:  35%|███▌      | 353/1007 [05:58<11:39,  1.07s/it]

In [12]:
# ── YOLO11s @ 640 — Artifact Retrieval ────────────────────────────────────
import wandb
from pathlib import Path

# Config for retrieval
MODEL_ID = "yolo11s_640"
RUN_NAME = f"{MODEL_ID}_baseline"
# Targeting Epoch 12 for the S model as it was the definitive peak
TARGET_EPOCH = 12
# Path derived from your W&B logs
ARTIFACT_PATH = f"bscs22030-information-technology-university/{WANDB_PROJECT}/{RUN_NAME}-epoch-{TARGET_EPOCH:03d}:latest"

print(f"📦 Attempting to download YOLO11s artifact: {ARTIFACT_PATH}")

try:
    api = wandb.Api()
    artifact = api.artifact(ARTIFACT_PATH)
    artifact_dir = artifact.download(root=f"./weights/{MODEL_ID}_e{TARGET_EPOCH}")
    
    # Path to the downloaded weight file
    BEST_WEIGHTS_PATH_S = Path(artifact_dir) / "last.pt"
    print(f"✅ Weights downloaded to: {BEST_WEIGHTS_PATH_S}")
except Exception as e:
    print(f"❌ Failed to download artifact: {e}")
    # Fallback to local best.pt if the run just finished
    BEST_WEIGHTS_PATH_S = Path("NPS-Drone-Baseline-Benchmark") / RUN_NAME / "weights" / "best.pt"
    print(f"⚠️ Falling back to local best weights: {BEST_WEIGHTS_PATH_S}")

📦 Attempting to download YOLO11s artifact: bscs22030-information-technology-university/NPS-Drone-Baseline-Benchmark/yolo11s_640_baseline-epoch-030:latest


wandb: Downloading large artifact 'yolo11s_640_baseline-epoch-030:latest', 54.48MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:01.1 (51.1MB/s)


✅ Weights downloaded to: weights/yolo11s_640_e30/last.pt


In [13]:
# ── Research Evaluation: YOLO11s Every-4th-Frame Protocol ──────────────────
import shutil
import os
import yaml
import wandb
from pathlib import Path
from ultralytics import YOLOBEST_WEIGHTS_PATH_S

# 1. Define Test Source
TEST_DIR = DATASET_ROOT / "images" / "test"

# 2. Setup Temporary Subset (Every 4th Frame)
every4th = get_every_4th_frame_paths(str(TEST_DIR))
evaluated_stems = {p.stem for p in every4th}

tmp_test_s_dir = Path("/kaggle/working/eval_s_every4th")
tmp_test_img_s = tmp_test_s_dir / "images"
tmp_test_lbl_s = tmp_test_s_dir / "labels"

tmp_test_img_s.mkdir(parents=True, exist_ok=True)
tmp_test_lbl_s.mkdir(parents=True, exist_ok=True)

for src in every4th:
    dst_img = tmp_test_img_s / src.name
    if not dst_img.exists():
        os.symlink(src.resolve(), dst_img)
    lbl_src = (DATASET_ROOT / "labels" / "test" / src.name).with_suffix(".txt")
    if lbl_src.exists():
        dst_lbl = tmp_test_lbl_s / lbl_src.name
        if not dst_lbl.exists():
            os.symlink(lbl_src.resolve(), dst_lbl)

# 3. Create Subset YAMLBEST_WEIGHTS_PATH_S
subset_yaml_s = {
    "path" : str(tmp_test_s_dir),
    "train": "images", "val": "images", "test" : "images",
    "nc": 1, "names": {0: "drone"},
}
SUBSET_YAML_S_PATH = "/kaggle/working/subset_test_eval_s.yaml"
with open(SUBSET_YAML_S_PATH, "w") as f:
    yaml.dump(subset_yaml_s, f)

# 4. Run Evaluation
eval_model_s = YOLO(str(BEST_WEIGHTS_PATH_S))
print(f"\n🚀 Evaluating YOLO11s (Epoch {TARGET_EPOCH}) on test set...")

test_results_s = eval_model_s.val(
    data      = SUBSET_YAML_S_PATH,
    split     = "test",
    imgsz     = CONFIG["imgsz"],
    conf      = CONFIG["conf"],
    iou       = CONFIG["iou"],
    save_json = True,
    save_txt  = True,
    plots     = True,
    verbose   = True,
)

# 5. Metrics Extraction & FPPI Calculation
metrics_s = extract_metrics(test_results_s, f"{MODEL_ID}_epoch_{TARGET_EPOCH}")

# FIX: Use test_results_s.save_dir directly
pred_lbl_dir_s = Path(test_results_s.save_dir) / "labels"

metrics_s["FPPI"] = compute_fppi(pred_lbl_dir_s, str(TEST_DIR), evaluated_stems)

# 6. Log to W&B
run = wandb.init(project=WANDB_PROJECT, name=f"{RUN_NAME}_FINAL_EVAL", job_type="evaluation", reinit=True)
wandb.log({f"final_test_s/{k}": v for k, v in metrics_s.items() if k != "model"})

# Upload plots
for plot_file in ["PR_curve.png", "F1_curve.png", "confusion_matrix.png"]:
    p_path = Path(test_results_s.save_dir) / plot_file
    if p_path.exists():
        wandb.log({f"plots_s/{plot_file.split('.')[0]}": wandb.Image(str(p_path))})

wandb.finish()

# 7. Final Table Update
ALL_RESULTS.append(metrics_s)
save_results_csv()
print_results_table()

# Cleanup
shutil.rmtree(tmp_test_s_dir, ignore_errors=True)
print(f"\n✅ Final Evaluation for {MODEL_ID} Complete.")

Test set  : 12355 total frames
Evaluating: 3089 frames (every 4th, skip=4)

🚀 Evaluating YOLO11s (Epoch 30) on test set...
Ultralytics 8.3.100 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


val: Scanning /kaggle/working/eval_s_every4th/labels... 3089 images, 809 backgrounds, 0 corrupt: 100%|██████████| 3089/3089 [00:18<00:00, 162.88it/s]


val: New cache created: /kaggle/working/eval_s_every4th/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 194/194 [00:54<00:00,  3.55it/s]


                   all       3089       4099       0.83      0.588      0.725      0.344
Speed: 0.2ms preprocess, 5.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Saving runs/detect/val4/predictions.json...
Results saved to runs/detect/val4

  ┌── yolo11s_640_epoch_30 ─────────────────────────
  │  model         : yolo11s_640_epoch_30
  │  AP@0.5        : 0.7247
  │  AP@0.5:0.95   : 0.3445
  │  Precision     : 0.8299
  │  Recall        : 0.5882
  │  F1            : 0.6885
  │  FPS           : 153.04
  │  inf_ms        : 5.6
  │  FPPI          : None
  └────────────────────────────────────────
  FPPI = 3.040142  (9391 FPs on 3089 empty frames / 3089 evaluated frames)


final_test_s/AP@0.5,▁
final_test_s/AP@0.5:0.95,▁
final_test_s/F1,▁
final_test_s/FPPI,▁
final_test_s/FPS,▁
final_test_s/Precision,▁
final_test_s/Recall,▁
final_test_s/inf_ms,▁
final_test_s/AP@0.5,0.7247
final_test_s/AP@0.5:0.95,0.3445
final_test_s/F1,0.6885


Results saved → /kaggle/working/baseline_results.csv

RESULTS vs TransVisDrone — NPS Dataset
                      model  AP@0.5  Precision  Recall      F1    FPS     FPPI
TransVisDrone (τ=5, 1280px)  0.9500     0.9200  0.9100     N/R  24.60 0.000437
       yolo11s_640_epoch_12  0.7285     0.8203  0.6014   0.694 154.23 4.146649
       yolo11s_640_epoch_30  0.7247     0.8299  0.5882  0.6885 153.04 3.040142

✅ Final Evaluation for yolo11s_640 Complete.
